<div style="background-color: #0F172A; padding: 20px; border-radius: 10px; border-left: 10px solid #6366F1; color: white;">
  <h1 style="color: #6366F1; margin: 0;">🌐 ARKOS MI — Etapa 2: Coleta de Dados via APIs e Web Scraping</h1>
  <h3 style="color: #10B981; margin-top: 5px;">Aquisição de Sinais Multicanal | Módulo Marketing Intelligence (MI)</h3>
  <p style="font-size: 1.1em; color: #F4F2ED;">
    Nesta etapa, construímos a <b>Engenharia de Dados</b> necessária para o sistema de Signal Intelligence. Coletamos tanto dados estruturados (APIs) quanto não estruturados (Web Scraping).
  </p>
</div>

In [1]:
import sys
import os
import requests
import pandas as pd
import plotly.express as px
from bs4 import BeautifulSoup

# Adicionando src ao path
sys.path.append(os.path.abspath('../src'))
from limpeza import limpar_para_spacy
from coleta import *

print("✅ Módulos de Coleta e Pandas PRONTOS!")

✅ Módulos de Coleta e Pandas PRONTOS!


<h2 style="color: #6366F1; border-bottom: 2px solid #6366F1;">🔗 SEÇÃO 1 — Web API com `requests`</h2>

Utilizamos a biblioteca `requests` para interagir com endpoints RESTful e validar a disponibilidade dos serviços.

In [2]:
# 1.1 Verificação de Status Code (GitHub API)
query = "fintech brazil"
url_gh = f"https://api.github.com/search/repositories?q={query}"
response = requests.get(url_gh)

print(f"📡 Status da Requisição GitHub: {response.status_code}")
if response.status_code == 200:
    # 1.2 Mostrar JSON Estruturado (Preview)
    data_gh = response.json()
    print(f"📦 Total de Repositórios Encontrados: {data_gh['total_count']}")
    repos_gh = data_gh['items'][:5] # Pegando os 5 primeiros
    display(pd.DataFrame(repos_gh)[['name', 'stargazers_count', 'description', 'updated_at']])

📡 Status da Requisição GitHub: 200
📦 Total de Repositórios Encontrados: 1


,name,stargazers_count,description,updated_at
0,brazilfinance,0,🚀 BrazilFinance - Democratizando dados finance...,2026-03-18T14:03:37Z


In [10]:
# 1.3 Coleta via BrasilAPI (Dados de Corretoras)
url_br = "https://brasilapi.com.br/api/cvm/corretoras/v1"
response_br = requests.get(url_br)

print(f"📡 Status da Requisição BrasilAPI: {response_br.status_code}")
if response_br.status_code == 200:
    dados_br = response_br.json()[:100] # Primeiras 100 corretoras
    df_br = pd.DataFrame(dados_br)
    print(f"✅ {len(df_br)} corretoras coletadas.")
else:
    # Fallback para o notebook não travar (NameError: df_br)
    print("⚠️ API indisponível ou URL incorreta. Criando DataFrame simulado.")
    mock_data = [
        {"nome_social": "Arkos Capital", "uf": "SP", "municipio": "São Paulo"},
        {"nome_social": "Fintech BR", "uf": "RJ", "municipio": "Rio de Janeiro"},
        {"nome_social": "Startup Broker", "uf": "SC", "municipio": "Florianópolis"}
    ]
    df_br = pd.DataFrame(mock_data)


📡 Status da Requisição BrasilAPI: 200
✅ 100 corretoras coletadas.


<h2 style="color: #6366F1; border-bottom: 2px solid #6366F1;">📊 SEÇÃO 2 — Estruturação e Limpeza com `pandas`</h2>

Transformamos JSON bruto em DataFrames limpos e prontos para análise.

In [11]:
# Limpeza de dados (Simulando colunas financeiras como solicitado na Aula 3)
import numpy as np

# Adicionando dados simulados para demonstração de limpeza (Patrimônio e Data)
df_br['patrimonio_liquido'] = np.random.uniform(1000000, 50000000, size=len(df_br))
df_br['data_fundacao'] = pd.to_datetime('2020-01-01') + pd.to_timedelta(np.random.randint(0, 1000, size=len(df_br)), unit='D')

# Técnicas de Limpeza pandas
df_br['patrimonio_liquido'] = pd.to_numeric(df_br['patrimonio_liquido'], errors='coerce')
df_clean = df_br.dropna(subset=['municipio', 'uf']).sort_values(by='patrimonio_liquido', ascending=False)

print("Top 5 Corretoras por Patrimônio Líquido (Simulado):")
display(df_clean[['nome_social', 'uf', 'municipio', 'patrimonio_liquido']].head(5))

Top 5 Corretoras por Patrimônio Líquido (Simulado):


,nome_social,uf,municipio,patrimonio_liquido
48,BANORTE CVM SA,PE,RECIFE,4.979494e+07
84,CARVALHO CCTM LTDA,BA,SALVADOR,4.945730e+07
26,ARIJU S/A CCTVM,RJ,RIO DE JANEIRO,4.913067e+07
92,CITY EMPREENDIMENTOS E SERVIÇOS LTDA.,RJ,RIO DE JANEIRO,4.806367e+07
14,AGUIAR CCVM LTDA,SP,SANTOS,4.803397e+07


<h2 style="color: #6366F1; border-bottom: 2px solid #6366F1;">📈 SEÇÃO 3 — Visualização Interativa com `Plotly`</h2>

Geração de indicadores visuais de tráfego e distribuição geográfica.

In [12]:
# 3.1 Gráfico de Bolhas (Patrimônio x Data x Região - Aula 3)
fig_bolhas = px.scatter(df_clean,
                        x='data_fundacao', 
                        y='patrimonio_liquido', 
                        size='patrimonio_liquido',
                        color='uf',
                        hover_name='nome_social',
                        title="ARKOS MI — Panorama de Corretoras (Patrimônio vs Tempo)",
                        template="plotly_dark",
                        color_discrete_sequence=px.colors.qualitative.Prism)
fig_bolhas.show()

# 3.2 Gráfico de Barras: Repositórios Fintech (GitHub Stars)
df_repos = pd.DataFrame(repos_gh)
fig_stars = px.bar(df_repos, x='name', y='stargazers_count', color='name',
                   title="Top Repositórios Fintech (Stars)", 
                   template="plotly_dark",
                   labels={'stargazers_count': 'Estrelas', 'name': 'Repositório'})
fig_stars.show()

<h2 style="color: #6366F1; border-bottom: 2px solid #6366F1;">🕸️ SEÇÃO 4 — BeautifulSoup (Web Scraping HTML)</h2>

Extração de dados semi-estruturados em sites que não oferecem API.

In [13]:
url_quotes = "http://quotes.toscrape.com/"
response_q = requests.get(url_quotes)
print(f"📡 Status Scraping: {response_q.status_code}")

if response_q.status_code == 200:
    soup = BeautifulSoup(response_q.text, "html.parser")
    quotes_list = []
    
    # Extraindo citações e autores
    for container in soup.find_all('div', class_='quote'):
        text = container.find('span', class_='text').text
        author = container.find('small', class_='author').text
        quotes_list.append({"Citação": text, "Autor": author})
    
    df_quotes = pd.DataFrame(quotes_list)
    print("Frases famosas coletadas via BeautifulSoup:")
    display(df_quotes.head(5))

📡 Status Scraping: 200
Frases famosas coletadas via BeautifulSoup:


,Citação,Autor
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


<h2 style="color: #6366F1; border-bottom: 2px solid #6366F1;">📦 SEÇÃO 5 — Construção do Corpus Digital</h2>

Consolidação final dos textos brutos limpos para alimentar os modelos da Etapa 3.

In [14]:
# Simulando a construção do corpus unificado
def construir_corpus(lista_df):
    corpus_final = []
    for df in lista_df:
        if 'description' in df.columns: # GitHub
            corpus_final.extend(df['description'].dropna().tolist())
        if 'Citação' in df.columns: # Scraping
            corpus_final.extend(df['Citação'].tolist())
    return corpus_final

corpus_bruto = construir_corpus([pd.DataFrame(repos_gh), df_quotes])

# Aplicando limpeza Regex antes do Processamento spaCy (Conexão Etapa 1 -> Etapa 2)
corpus_limpo = [limpar_para_spacy(txt) for txt in corpus_bruto[:10]]

print(f"✅ Corpus construído com {len(corpus_limpo)} documentos limpos.")
print("Amostra do Corpus para Etapa 3:")
print(corpus_limpo[0][:100] + "...")

# Salvando como referência para a Etapa 3
%store corpus_limpo

✅ Corpus construído com 10 documentos limpos.
Amostra do Corpus para Etapa 3:
brazilfinance  democratizando dados financeiros brasileiros via google sheets. api unificada bcb  fo...
Stored 'corpus_limpo' (list)
